<a href="https://colab.research.google.com/github/amarchini5339/AAI-511-group7/blob/fix-dataset/notebooks/Notebook-Team_7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Environment Setup and Library Imports

Set up Colab environment and import preliminary required packages

Set up Google Colab

In [1]:
from google.colab import drive
drive.mount('/content/drive')

# Path to extracted features (adjust as necessary)
DATA_PATH = "/content/drive/MyDrive/AAI-511-group7-main/notebooks"

Mounted at /content/drive


Install necessary libraries (through Google Colab)

In [2]:
%pip install -q pretty_midi numpy tensorflow scikit-learn matplotlib seaborn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 23.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 kB 2.5 MB/s eta 0:00:00


Import required Python libraries

In [3]:
import pretty_midi
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import os
from tqdm import tqdm
from sklearn.utils.class_weight import compute_class_weight
from collections import Counter
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold, StratifiedKFold

# Feature Extraction

### MIDI Dataset Preprocessing for LSTM

**Summary:**

This code cell implements a data preprocessing pipeline for a MIDI dataset, preparing it for an LSTM model. It defines a function to extract features (pitch, velocity, duration, time since last note) from MIDI files. It then iterates through train, dev, and test directories organized by composer, extracts features for allowed composers, normalizes sequence lengths by truncating or padding, and assigns integer labels. Finally, it converts the processed features and labels into NumPy arrays for each data split.

**Key Tasks:**

-   **Feature Extraction:** Extracts pitch, velocity, duration, and time since last note for each note in a MIDI file.
-   **Sequence Normalization:** Truncates or pads note sequences to a fixed length.
-   **Data Organization:** Maps composer names to integer labels and organizes features and labels by data split.
-   **Data Consolidation:** Converts lists of features and labels into NumPy arrays.

The output provides shapes of the prepared data and the label-to-composer mapping, indicating the dataset is ready for model training.

In [4]:
DATA_DIR = "/content/drive/MyDrive/AAI-511-group7-main/Composer_Dataset"#../Composer_Dataset/NN_midi_files_extended"#"/content/drive/MyDrive/AAI-511-group7-main/Composer_Dataset"
SETS = ['train', 'dev', 'test']
FIXED_SEQUENCE_LENGTH = 5000  # truncate or pad to this many notes
ALLOWED_COMPOSERS = {'bach', 'beethoven', 'chopin', 'mozart'} # specified in project requirements
DOWNSAMPLE = {'bach': 0.5}

def extract_lstm_features(midi_path, max_length=FIXED_SEQUENCE_LENGTH):
    try:
        midi = pretty_midi.PrettyMIDI(midi_path)
        notes = []
        for inst in midi.instruments:
            if inst.is_drum:
                continue
            notes.extend(inst.notes)
        if not notes:
            return None  # skip empty or drum-only files

        notes.sort(key=lambda n: n.start)
        sequence = []
        prev_start = 0.0
        for note in notes:
            pitch = note.pitch
            velocity = note.velocity
            duration = note.end - note.start
            time_since_last = note.start - prev_start
            sequence.append([pitch, velocity, duration, time_since_last])
            prev_start = note.start

        # Zero-padding instead of repetition
        if len(sequence) >= max_length:
            sequence = sequence[:max_length]
        else:
            pad_length = max_length - len(sequence)
            padding = [[0.0, 0.0, 0.0, 0.0]] * pad_length
            sequence.extend(padding)

        return np.array(sequence, dtype=np.float32)

    except Exception as e:
        print(f"Failed to process {midi_path}: {e}")
        return None

MIDI_EXTS = {".mid", ".midi", ".MID", ".MIDI"}
RNG_SEED = 42
SPLIT_RATIOS = {"train": 0.70, "dev": 0.15, "test": 0.15}

# --- Helpers ---
def iter_mid_paths(root_dir):
    """Yield .mid/.midi files recursively under root_dir."""
    for dirpath, _, filenames in os.walk(root_dir):
        for fn in filenames:
            _, ext = os.path.splitext(fn)
            if ext in MIDI_EXTS:
                yield os.path.join(dirpath, fn)

def safe_stack(lst, split_name):
    if len(lst) == 0:
        raise ValueError(f"No data collected for split '{split_name}'. "
                         f"Check your paths/filters.")
    return np.stack(lst)

# --- Build splits from composer folders (no train/dev/test dirs) ---
rng = np.random.RandomState(RNG_SEED)

# Stable label mapping (alphabetical over allowed set)
composer_to_idx = {c: i for i, c in enumerate(sorted(ALLOWED_COMPOSERS))}

# Collect all file paths per composer (recursive)
files_by_composer = {}
for composer_name in os.listdir(DATA_DIR):
    cname = composer_name.lower()
    if cname not in ALLOWED_COMPOSERS:
        continue
    cpath = os.path.join(DATA_DIR, composer_name)
    if not os.path.isdir(cpath):
        continue
    midi_files = list(iter_mid_paths(cpath))
    if midi_files:
        files_by_composer[cname] = midi_files

if not files_by_composer:
    raise ValueError("No MIDI files found. Check DATA_DIR and folder structure.")

# Prepare containers
X = {'train': [], 'dev': [], 'test': []}
y = {'train': [], 'dev': [], 'test': []}

def split_counts(n, train_ratio, dev_ratio):
    """Compute per-composer counts with sane rounding and edge-case handling."""
    if n <= 2:
        # tiny classes: put at least 1 in train, remainder in test
        train_n = 1 if n >= 1 else 0
        dev_n = 0
        test_n = max(0, n - train_n)
        return train_n, dev_n, test_n
    train_n = int(round(n * train_ratio))
    dev_n = int(round(n * dev_ratio))
    # Ensure at least 1 in train if any exist
    if train_n == 0 and n > 0:
        train_n = 1
    # Fix rounding overflow/underflow
    if train_n + dev_n > n - 1:  # leave at least 1 for test if possible
        overflow = train_n + dev_n - (n - 1)
        # trim dev first, then train
        take_from_dev = min(dev_n, overflow)
        dev_n -= take_from_dev
        overflow -= take_from_dev
        if overflow > 0:
            train_n = max(1, train_n - overflow)
    test_n = n - train_n - dev_n
    return train_n, dev_n, test_n

# Do per-composer splits, then preprocess each file
for cname, paths in files_by_composer.items():
    label = composer_to_idx[cname]
    paths = np.array(paths)
    perm = rng.permutation(len(paths))
    paths = paths[perm]

    if cname in DOWNSAMPLE:
        keep_n = max(1, int(len(paths) * DOWNSAMPLE[cname]))
        paths = paths[:keep_n]
        print(f"Downsampled {cname}: keeping {keep_n} files")

    t_ratio = SPLIT_RATIOS['train']
    d_ratio = SPLIT_RATIOS['dev']
    train_n, dev_n, test_n = split_counts(len(paths), t_ratio, d_ratio)

    train_paths = paths[:train_n]
    dev_paths   = paths[train_n:train_n+dev_n]
    test_paths  = paths[train_n+dev_n:]

    for split, split_paths in [('train', train_paths), ('dev', dev_paths), ('test', test_paths)]:
        print(f"Processing {split.upper()} for {cname} ({len(split_paths)} files)...")
        for midi_path in tqdm(split_paths, desc=f"{split}/{cname}", leave=False):
            features = extract_lstm_features(midi_path)
            if features is not None:
                X[split].append(features)
                y[split].append(label)

# Convert to numpy arrays
for split in ['train', 'dev', 'test']:
    X[split] = safe_stack(X[split], split)
    y[split] = np.array(y[split], dtype=np.int64)
    print(f"{split}: {X[split].shape}, labels: {y[split].shape}")

print("Composer to index mapping (stable order):", composer_to_idx)

Processing TRAIN for mozart (180 files)...


train/mozart:  14%|█▍        | 26/180 [00:19<01:13,  2.10it/s]/usr/local/lib/python3.11/dist-packages/pretty_midi/pretty_midi.py:100: RuntimeWarning: Tempo, Key or Time signature change events found on non-zero tracks.  This is not a valid type 0 or type 1 MIDI file.  Tempo, Key or Time Signature may be wrong.
  warnings.warn(
train/mozart:  37%|███▋      | 67/180 [00:37<00:44,  2.54it/s]

Failed to process /content/drive/MyDrive/AAI-511-group7-main/Composer_Dataset/Mozart/Piano Sonatas/Nueva carpeta/K281 Piano Sonata n03 3mov.mid: Could not decode key with 2 flats and mode 2


Processing DEV for mozart (39 files)...


Processing TEST for mozart (38 files)...


...

Processing TEST for beethoven (32 files)...


Downsampled bach: keeping 512 files
Processing TRAIN for bach (358 files)...


Processing DEV for bach (77 files)...


Processing TEST for bach (77 files)...


train: (781, 5000, 4), labels: (781,)
dev: (167, 5000, 4), labels: (167,)
test: (168, 5000, 4), labels: (168,)
Composer to index mapping (stable order): {'bach': 0, 'beethoven': 1, 'chopin': 2, 'mozart': 3}


Saves the extracted LSTM features and labels for each data split (train, dev, test) as `.npy` files using NumPy's `np.save` function. This allows for efficient storage and later loading of preprocessed data, so feature extraction does not need to be repeated each time subsequent code is run. The files are named according to their split and stored in the current working directory.

In [5]:
# Set your local path to the data directory
DATA_PATH = "/content/drive/MyDrive/AAI-511-group7-main/notebooks"

# Save as .npy
np.save(os.path.join(DATA_PATH, "X_train_lstm.npy"), X['train'])
np.save(os.path.join(DATA_PATH, "y_train_lstm.npy"), y['train'])
np.save(os.path.join(DATA_PATH, "X_dev_lstm.npy"), X['dev'])
np.save(os.path.join(DATA_PATH, "y_dev_lstm.npy"), y['dev'])
np.save(os.path.join(DATA_PATH, "X_test_lstm.npy"), X['test'])
np.save(os.path.join(DATA_PATH, "y_test_lstm.npy"), y['test'])

# Model Building and Training

## Initial Data Loading

This code cell loads the pre-processed data, which has previously been saved as NumPy files (`.npy`) and a Python dictionary (`.pkl`). The data consists of musical features (`X`) and corresponding composer labels (`y`) for training, development, and testing sets.

The following steps are performed:

1.  **Set Data Path**: The `DATA_PATH` variable is defined to point to the location of the data files.
2.  **Load Raw Data**: The feature and label arrays (`X_train`, `y_train`, etc.) are loaded from their respective `.npy` files. The original `composer_to_idx` dictionary, which maps all composer names to integer indices, is loaded from a `.pkl` file.
3.  **Define and Filter Data**: A new `composer_to_idx` dictionary is created that only includes the four target composers: Bach, Beethoven, Chopin, and Mozart. The `filter_data` function then iterates through the loaded datasets, retaining only the examples that correspond to these four composers and re-mapping their labels to the new index scheme.
4.  **Confirm Shapes**: The shapes of the filtered training, development, and testing arrays are printed to confirm that the data has been loaded and filtered correctly.
5.  **Create Inverse Mapping**: A new `idx_to_composer` dictionary is created to allow for easy mapping from the new integer indices back to the names of the four target composers.

In [6]:
# Load .npy data files
X_train = np.load(os.path.join(DATA_PATH, "X_train_lstm.npy"))
y_train = np.load(os.path.join(DATA_PATH, "y_train_lstm.npy"))
X_dev   = np.load(os.path.join(DATA_PATH, "X_dev_lstm.npy"))
y_dev   = np.load(os.path.join(DATA_PATH, "y_dev_lstm.npy"))
X_test  = np.load(os.path.join(DATA_PATH, "X_test_lstm.npy"))
y_test  = np.load(os.path.join(DATA_PATH, "y_test_lstm.npy"))

# Confirm loading shapes
print("Loaded and filtered feature and label arrays with shapes:")
print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_dev:", X_dev.shape, "y_dev:", y_dev.shape)
print("X_test:", X_test.shape, "y_test:", y_test.shape)

# Update idx_to_composer for the filtered set
idx_to_composer = {v: k for k, v in composer_to_idx.items()}

Loaded and filtered feature and label arrays with shapes:
X_train: (781, 5000, 4) y_train: (781,)
X_dev: (167, 5000, 4) y_dev: (167,)
X_test: (168, 5000, 4) y_test: (168,)


## LSTM Model Building

The LSTM model is designed to classify musical scores by composer, leveraging the sequential nature of MIDI feature data. The architecture consists of the following layers:

- **Masking Layer:** Handles variable-length input sequences by ignoring padded values (set to zero), ensuring the model only processes meaningful musical information.
- **LSTM Layer:** Captures long-term dependencies and temporal patterns in the music, making it well-suited for sequential MIDI note data.
- **Dense Layer (ReLU):** Introduces non-linearity and enables the model to learn complex feature representations from the LSTM output.
- **Dropout Layer:** Provides regularization by randomly dropping units during training, which helps prevent overfitting.
- **Output Dense Layer (Softmax):** Produces a probability distribution over all composer classes, supporting

In [7]:
num_classes = len(np.unique(y_train))
input_shape = X_train.shape[1:]  # (5000, 4)

lstm_model = keras.Sequential([
    layers.Masking(mask_value=0., input_shape=input_shape),
    layers.LSTM(128, return_sequences=False),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(num_classes, activation='softmax')
])

lstm_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
lstm_model.summary()


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/masking.py:48: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ masking (Masking)               │ (None, 5000, 4)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 128)            │        68,096 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │           260 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 76,612 (299.27 KB)

 Trainable params: 76,612 (299.27 KB)

 Non-trainable params: 0 (0.00 B)

## LSTM Model Training

The LSTM model is trained to classify musical scores by composer using the extracted MIDI features.

- **Training and Validation Split:**  
  The model is trained using a standard split of training and validation data, completed in preceding code. This allows for representative evaluation of the model’s performance during training and helps monitor for overfitting or underfitting.

- **Early Stopping:**  
  To ensure robust generalization and prevent overfitting, early stopping is used. This mechanism monitors the validation loss and halts training when no further improvement is observed, restoring the best weights.

- **Performance Tracking:**  
  Both training and validation loss and accuracy are tracked throughout the process. This provides insight into the model's learning dynamics and helps identify potential issues.

- **Goal:**  
  The approach ensures that the final model achieves optimal performance on unseen data while maintaining generalizability across different composers.

In [8]:
# Callback for early stopping
early_stopping = EarlyStopping(
    monitor='val_loss', patience=5, restore_best_weights=True
)

# Model checkpoint to save the best model
checkpoint = ModelCheckpoint(
    "best_lstm_model.keras", monitor='val_loss', save_best_only=True, save_weights_only=False
)

# Compute class weights since Bach has many more samples than the other composers
classes = np.unique(y_train)
weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
class_weight = {int(c): float(w) for c, w in zip(classes, weights)}
print("Class weights:", class_weight)

# Train the model
history = lstm_model.fit(
    X_train, y_train,
    epochs=40,
    batch_size=16,
    validation_data=(X_dev, y_dev),
    callbacks=[early_stopping, checkpoint],
    class_weight=class_weight
)

Class weights: {0: 0.5453910614525139, 1: 1.3104026845637584, 2: 2.055263157894737, 3: 1.0907821229050279}
Epoch 1/40
49/49 ━━━━━━━━━━━━━━━━━━━━ 176s 4s/step - accuracy: 0.3901 - loss: 1.3751 - val_accuracy: 0.5689 - val_loss: 1.1852
Epoch 2/40
49/49 ━━━━━━━━━━━━━━━━━━━━ 199s 3s/step - accuracy: 0.5059 - loss: 1.2506 - val_accuracy: 0.4850 - val_loss: 1.1186
Epoch 3/40
47/49 ━━━━━━━━━━━━━━━━━━━━ 6s 3s/step - accuracy: 0.5248 - loss: 1.1812

KeyboardInterrupt: 

### Display training and validation curves

Plotting these curves helps assess the model's learning behavior, diagnose overfitting or underfitting, and determine if early stopping was triggered at an appropriate point. By comparing the trends of training and validation metrics, we can better understand how well the model generalizes to unseen data.

In [ ]:
# Plot training & validation loss and accuracy
plt.figure(figsize=(12, 4))

# Loss curve
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

# Accuracy curve
plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.tight_layout()
plt.show()

## LSTM Model Evaluation

The evaluation metrics chosen for the LSTM model are:

- **Accuracy:** Measures the overall proportion of correctly classified samples, providing a general sense of model effectiveness.
- **Precision:** Calculated using the weighted average across all classes, precision reflects how many predicted composers are correct.
- **Recall:** Also weighted across classes, recall indicates how many actual composers are correctly identified.
- **F1-Score:** Combines precision and recall into a single metric, balancing both false positives and false negatives.

Additional evaluation tools include:

- **Classification Report:** Provides per-class precision, recall, and F1-scores, highlighting which composers are most and least accurately classified.
- **Confusion Matrix:** Visualizes the distribution of correct and incorrect predictions across all classes, making it easier to identify specific patterns of misclassification.

In [ ]:
# Evaluate on test set
test_loss, test_accuracy = lstm_model.evaluate(X_test, y_test, verbose=0)
print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Test Loss: {test_loss:.4f}")

# Predictions
y_pred_probs = lstm_model.predict(X_test, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)

# Metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')

print("\nDetailed Test Metrics:")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")

# Classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix - LSTM Composer Classification')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()

In [ ]:
class SparseCategoricalFocalLoss(tf.keras.losses.Loss):
    def __init__(self, num_classes, gamma=2.0, alpha=None, from_logits=False, name="sparse_categorical_focal_loss"):
        super().__init__(name=name)
        self.num_classes = num_classes
        self.gamma = gamma
        self.from_logits = from_logits
        if alpha is None:
            self.alpha = None
        else:
            alpha = tf.convert_to_tensor(alpha, dtype=tf.float32)
            self.alpha = tf.fill([num_classes], alpha) if alpha.shape.rank == 0 else alpha

    def call(self, y_true, y_pred):
        y_pred = tf.nn.softmax(y_pred, axis=-1) if self.from_logits else tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        y_true_oh = tf.one_hot(tf.cast(y_true, tf.int32), depth=self.num_classes)
        pt = tf.reduce_sum(y_true_oh * y_pred, axis=-1)
        alpha_t = tf.reduce_sum(y_true_oh * self.alpha, axis=-1) if self.alpha is not None else 1.0
        loss = -alpha_t * tf.pow(1.0 - pt, self.gamma) * tf.math.log(pt)
        return tf.reduce_mean(loss)

lstm_model = keras.Sequential([
    layers.Masking(mask_value=0., input_shape=input_shape),
    layers.LSTM(128, return_sequences=True),
    layers.Dropout(0.2),
    layers.LSTM(64, return_sequences=True),
    layers.Dropout(0.2),
    layers.LSTM(32, return_sequences=False),
    layers.Dense(num_classes, activation='softmax')
])

counts = np.bincount(y_train, minlength=num_classes).astype(np.float32)
inv_freq = 1.0 / np.maximum(counts, 1.0)
alpha_vec = inv_freq / np.sum(inv_freq) * num_classes

opt = tf.keras.optimizers.AdamW(learning_rate=1e-3, weight_decay=1e-4, clipnorm=1.0)
lstm_model.compile(optimizer=opt,
                   loss=SparseCategoricalFocalLoss(num_classes=num_classes, gamma=2.0, alpha=alpha_vec, from_logits=False),
                   metrics=['accuracy'])
lstm_model.summary()

In [ ]:
history = lstm_model.fit(
    X_train, y_train,
    epochs=40,
    batch_size=16,
    validation_data=(X_dev, y_dev),
    callbacks=[early_stopping, checkpoint]
)

In [ ]:
# Plot training & validation loss and accuracy
plt.figure(figsize=(12, 4))

# Loss curve
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

# Accuracy curve
plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Evaluate on test set
test_loss, test_accuracy = lstm_model.evaluate(X_test, y_test, verbose=0)
print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Test Loss: {test_loss:.4f}")

# Predictions
y_pred_probs = lstm_model.predict(X_test, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)

# Metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')

print("\nDetailed Test Metrics:")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")

# Classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix - LSTM Composer Classification')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()

### LSTM Cross-Validation
The above metrics show an accuracy, precision, recall, and F1 of 1.0. While this may seem like great performance, in the world of artificial intelligence, such high performance is typically a sign that something has gone wrong. To check whether model is simply "memorizing" or truly learning, the following code cell implements k-fold cross-validation to evaluate the initial LSTM model more robustly, especially given the limited dataset size. Here's how it works:

1. Import Libraries: Imports necessary libraries from scikit-learn (for KFold) and TensorFlow/Keras (for building and training the model).
2. Combine Data: It concatenates the training and development feature (X_train, X_dev) and label (y_train, y_dev) arrays into combined datasets (X_combined, y_combined). This combined data will be used for cross-validation.
3. Define KFold: KFold is initialized with n_splits=5 (meaning the data will be divided into 5 folds), shuffle=True (to randomly shuffle the data before splitting), and random_state=42 (for reproducibility).
4. Initialize Storage: Empty lists fold_accuracies and fold_losses are created to store the evaluation results for each fold.
5. Iterate Through Folds: The code loops through each fold generated by kf.split(). In each iteration:
  - train_index and val_index contain the indices for the training and validation data for the current fold.
  - The data is split into X_train_fold, y_train_fold (for training in this fold) and X_val_fold, y_val_fold (for validation in this fold) using the indices.
  - Build New Model: A new instance of the LSTM model (lstm_model_cv) is created and compiled from scratch. In cross-validation, this prevents data leakage between folds (i.e., preventing the model from learning anything about a validation fold from previous training on that same data if the same model instance were reused). The model architecture is the same as the initial LSTM model.
  - Define Early Stopping: A new EarlyStopping callback is defined for each fold to monitor validation loss and stop training if it plateaus, restoring the best weights for that fold.
  - Train Model: The new model instance is trained on the X_train_fold and y_train_fold data, with X_val_fold and y_val_fold used for validation during training.
  - Evaluate Fold: The trained model is evaluated on the X_val_fold and y_val_fold to get the loss and accuracy for that specific fold.
  - The fold's results are printed and appended to the fold_losses and fold_accuracies lists.
6. Report Average Results: After iterating through all folds, the code calculates and prints the average validation accuracy and loss across all folds using np.mean().

This process provides a more reliable measure of the initial LSTM model's performance by averaging results over multiple independent validation sets, helping to determine how well the model is likely to perform on unseen data in general, rather than just one specific test split.


In [ ]:
X_combined = np.concatenate([X_train, X_dev], axis=0)
y_combined = np.concatenate([y_train, y_dev], axis=0)
num_classes = len(np.unique(y_combined))
input_shape = X_combined.shape[1:]  # (5000, 4)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

fold_acc, fold_loss = [], []
all_true, all_pred = [], []

print("Running 5-fold stratified CV...")
for fold, (tr_idx, val_idx) in enumerate(skf.split(X_combined, y_combined), 1):
    print(f"\n--- Fold {fold}/5 ---")
    tf.keras.backend.clear_session()  # avoid graph/callback state carryover

    X_tr, X_val = X_combined[tr_idx], X_combined[val_idx]
    y_tr, y_val = y_combined[tr_idx], y_combined[val_idx]

    # Per-fold alpha for focal loss
    counts = np.bincount(y_tr, minlength=num_classes).astype(np.float32)
    inv = 1.0 / np.maximum(counts, 1.0)
    alpha_vec = inv / np.mean(inv)

    # Fresh model and optimizer per fold
    model = keras.Sequential([
        layers.Masking(mask_value=0., input_shape=input_shape),
        layers.LSTM(128, return_sequences=True),
        layers.Dropout(0.2),
        layers.LSTM(64, return_sequences=True),
        layers.Dropout(0.2),
        layers.LSTM(32, return_sequences=False),
        layers.Dense(num_classes, activation='softmax')
    ])

    opt = tf.keras.optimizers.AdamW(learning_rate=1e-3, weight_decay=1e-4, clipnorm=1.0)
    model.compile(
        optimizer=opt,
        loss=SparseCategoricalFocalLoss(num_classes=num_classes, gamma=2.0, alpha=alpha_vec, from_logits=False),
        metrics=['accuracy']
    )

    es = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
    ckpt = ModelCheckpoint("best_lstm_model.keras", monitor='val_loss', save_best_only=True, save_weights_only=False)

    history = model.fit(
        X_tr, y_tr,
        validation_data=(X_val, y_val),
        epochs=40,
        batch_size=16,
        callbacks=[es, ckpt],
        verbose=1
    )

    l, a = model.evaluate(X_val, y_val, verbose=0)
    fold_loss.append(l); fold_acc.append(a)

    y_prob = model.predict(X_val, verbose=0)
    y_hat = np.argmax(y_prob, axis=1)
    all_true.append(y_val); all_pred.append(y_hat)

print("\n--- Cross-Validation Results ---")
print(f"Avg Val Accuracy: {np.mean(fold_acc):.4f} ± {np.std(fold_acc):.4f}")
print(f"Avg Val Loss:     {np.mean(fold_loss):.4f} ± {np.std(fold_loss):.4f}")

### LSTM Cross-Validation Evaluation

The above "average validation accuracy" of 0.9119 is imperfect, but still good, which is much more believable than the previous value of 1.0. The below cell calculates and prints more evaluation measures and metrics to further examine the performance of the LSTM model.

In [ ]:
y_true_all = np.concatenate(all_true)
y_pred_all = np.concatenate(all_pred)
num_classes = int(y_true_all.max()) + 1
labels = list(range(num_classes))

print("\n--- Detailed Metrics Per Fold ---")
for i, (y_t, y_p) in enumerate(zip(all_true, all_pred), start=1):
    print(f"\n--- Fold {i} Classification Report ---")
    print(classification_report(y_t, y_p, labels=labels, zero_division=0, digits=4))
    # Optional: per-fold confusion matrix
    # print(confusion_matrix(y_t, y_p, labels=labels))

print("\n--- Overall Metrics on Concatenated Folds ---")
weighted_precision = precision_score(y_true_all, y_pred_all, average='weighted', zero_division=0)
weighted_recall    = recall_score(y_true_all, y_pred_all,   average='weighted', zero_division=0)
weighted_f1        = f1_score(y_true_all, y_pred_all,       average='weighted', zero_division=0)
overall_accuracy   = accuracy_score(y_true_all, y_pred_all)

print(f"Weighted Precision: {weighted_precision:.4f}")
print(f"Weighted Recall:    {weighted_recall:.4f}")
print(f"Weighted F1-Score:  {weighted_f1:.4f}")
print(f"Overall Accuracy:   {overall_accuracy:.4f}")

print("\n--- Overall Classification Report (Concatenated) ---")
print(classification_report(y_true_all, y_pred_all, labels=labels, zero_division=0, digits=4))

# (Optional) Fold-averaged metrics (mean ± std across folds)
fold_prec = [precision_score(t, p, average='weighted', zero_division=0) for t, p in zip(all_true, all_pred)]
fold_rec  = [recall_score(   t, p, average='weighted', zero_division=0) for t, p in zip(all_true, all_pred)]
fold_f1   = [f1_score(       t, p, average='weighted', zero_division=0) for t, p in zip(all_true, all_pred)]
fold_acc  = [accuracy_score( t, p)                                   for t, p in zip(all_true, all_pred)]

print("\n--- Fold-Averaged Metrics (mean ± std) ---")
print(f"Weighted Precision: {np.mean(fold_prec):.4f} ± {np.std(fold_prec):.4f}")
print(f"Weighted Recall:    {np.mean(fold_rec):.4f} ± {np.std(fold_rec):.4f}")
print(f"Weighted F1-Score:  {np.mean(fold_f1):.4f} ± {np.std(fold_f1):.4f}")
print(f"Accuracy:           {np.mean(fold_acc):.4f} ± {np.std(fold_acc):.4f}")

### LSTM Model Conclusion

The evaluation of the initial LSTM model highlighted promising performance on the small, 12-sample test set, achieving perfect accuracy, precision, recall, and F1-score (1.00). However, this result raised concerns about potential overfitting given the limited dataset size.

To obtain a more robust assessment, k-fold cross-validation was performed on the combined training and development data. **The cross-validation results provide a more realistic and reliable assessment.** While the model is **performing very well overall (average metrics around 0.91)** across different data splits, **the perfect score on the initial test set was likely an artifact of its small size.**

**The cross-validation metrics suggest the model is generalizing reasonably well on this dataset, but the performance is not uniformly perfect across all classes and data splits.** The variation in per-fold metrics and the slightly lower average scores compared to the initial test set evaluation lend some support to **our initial concern about potential overfitting due to the small dataset.** However, the cross-validation process itself appears to be functioning correctly without obvious signs of data leakage within the folds.

In conclusion, the cross-validation confirms that the LSTM model is capable of good performance on this composer classification task. While it doesn't achieve perfect scores as initially seen, the average metrics around 0.91 indicate solid generalization on the available data, providing a more trustworthy estimate of its performance on unseen musical pieces from these composers.

## CNN Model Building and Training

**Summary:**

This section constructs a 1D Convolutional Neural Network (CNN) as an alternative architecture for classifying composer sequences. The CNN is designed to process sequential MIDI feature data and extract local temporal patterns.

**Layer Structure:**

- `Conv1D` → `MaxPooling1D` → `Conv1D` → `MaxPooling1D` → `Flatten` → `Dense` → `Dropout` → `Dense (Softmax)`

**Training Approach:**

- The CNN model is compiled and trained using the same methodology as the LSTM model, allowing for direct performance comparison.

**Design Decisions:**

- CNNs are evaluated for their ability to capture local patterns in the extracted music features and their robustness with high-dimensional sequential inputs.
- This architecture is included to compare against the LSTM model as part of experimentation and ablation studies.

**Methodology Mapping:**

- **Model Building/Training:** Implements the methodology's directive to develop both LSTM

In [ ]:
cnn_model = keras.Sequential([
    layers.Conv1D(64, kernel_size=7, activation='relu', input_shape=input_shape),
    layers.MaxPooling1D(pool_size=2),
    layers.Conv1D(128, kernel_size=7, activation='relu'),
    layers.MaxPooling1D(pool_size=2),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(num_classes, activation='softmax')
])

cnn_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

cnn_model.summary()

# Train
history_cnn = cnn_model.fit(
    X_train, y_train,
    epochs=30,
    batch_size=16,
    validation_data=(X_dev, y_dev),
    callbacks=[early_stopping],
    verbose=0   # No verbose output for cleaner logs
)


## Display Training and Validation Curves

In [ ]:
import matplotlib.pyplot as plt

# Plot training & validation loss and accuracy
plt.figure(figsize=(12, 4))

# Loss curve
plt.subplot(1, 2, 1)
plt.plot(history_cnn.history['loss'], label='Training Loss')
plt.plot(history_cnn.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

# Accuracy curve
plt.subplot(1, 2, 2)
plt.plot(history_cnn.history['accuracy'], label='Training Accuracy')
plt.plot(history_cnn.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.tight_layout()
plt.show()

## CNN Model Evaluation

**Summary:**

This section evaluates the performance of the CNN model on the test set using standard classification metrics.

**Design Decisions:**

- The evaluation metrics and methodology are kept consistent with those used for the LSTM model, enabling a direct and fair comparison between the two architectures.
- This approach helps determine which model is better suited for composer classification based on the extracted MIDI features.

**Methodology Mapping:**

- **Model Evaluation:** Applies accuracy, precision, recall, F1-score, and classification reports to assess model performance.
- **Model Optimization:** Informs further tuning or selection of the most effective architecture for the task.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
import seaborn as sns
import matplotlib.pyplot as plt

# Evaluate on test set
test_loss_cnn, test_acc_cnn = cnn_model.evaluate(X_test, y_test, verbose=0)
print(f"Test Accuracy (CNN): {test_acc_cnn:.4f}")
print(f"Test Loss (CNN): {test_loss_cnn:.4f}")

# Predictions
y_pred_probs_cnn = cnn_model.predict(X_test, verbose=0)
y_pred_cnn = np.argmax(y_pred_probs_cnn, axis=1)

# Metrics
accuracy_cnn = accuracy_score(y_test, y_pred_cnn)
precision_cnn = precision_score(y_test, y_pred_cnn, average='weighted')
recall_cnn = recall_score(y_test, y_pred_cnn, average='weighted')
f1_cnn = f1_score(y_test, y_pred_cnn, average='weighted')

print("\nDetailed Test Metrics (CNN):")
print(f"Accuracy:  {accuracy_cnn:.4f}")
print(f"Precision: {precision_cnn:.4f}")
print(f"Recall:    {recall_cnn:.4f}")
print(f"F1-Score:  {f1_cnn:.4f}")

# Classification report
print("\nClassification Report (CNN):")
print(classification_report(y_test, y_pred_cnn))

# Confusion matrix
cm_cnn = confusion_matrix(y_test, y_pred_cnn)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_cnn, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix - CNN Composer Classification')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()

## Model Comparison: LSTM vs CNN

Compare models and respective performance here

# Model Optimization

### LSTM Optimization

LSTM Model Definition and Training

This code cell defines and trains a new Keras Sequential model, specifically a deep Long Short-Term Memory (LSTM) network, for a sequence classification task.

1. Model Architecture: It constructs a sequential model using a variety of layers.

  - Masking Layer: An initial Masking layer is used to ignore the padded values in the input sequences (mask_value=0.).

  - Bidirectional LSTM Layers: Two Bidirectional(LSTM) layers process the sequences. This approach allows the model to capture dependencies from both past and future timesteps, which is highly effective for sequential data.

  - Batch Normalization: BatchNormalization layers are included after each LSTM to stabilize and accelerate the training process.

  - Dense and Dropout Layers: A Dense layer with a ReLU activation function processes the LSTM output, followed by a Dropout layer for regularization to prevent overfitting.

  - Output Layer: The final Dense layer uses a softmax activation to output probabilities for the num_classes, enabling classification.

2. Model Compilation: The model is compiled with the Adam optimizer, the sparse_categorical_crossentropy loss function (suitable for multi-class classification with integer labels), and accuracy as the evaluation metric.

3. Model Training: The fit method is called to train the model using the preprocessed X_train and y_train data, with validation performed on X_dev and y_dev. An early_stopping callback is used to prevent overfitting by halting training when the validation loss stops improving.

In [ ]:
from tensorflow.keras import regularizers
from tensorflow.keras.layers import Bidirectional, Dense, Dropout, Masking, BatchNormalization
from tensorflow.keras.optimizers import Adam

# LSTM model
lstm_model_new = keras.Sequential([
    layers.Masking(mask_value=0., input_shape=input_shape),
    layers.Bidirectional(layers.LSTM(128, return_sequences=True)), # Reverted units
    layers.BatchNormalization(),
    layers.Bidirectional(layers.LSTM(64, return_sequences=False)), # Reverted units
    layers.BatchNormalization(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(num_classes, activation='softmax')
])

# Compile the model
lstm_model_new.compile(
    optimizer=Adam(),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Train the model (use same callbacks as before)
history_lstm_new = lstm_model_new.fit(
    X_train, y_train,
    epochs=30,
    batch_size=16,
    validation_data=(X_dev, y_dev),
    callbacks=[early_stopping],
    verbose=0   # No verbose output for cleaner logs
)

Save the model

In [ ]:
# Save the model
lstm_model_new.save(os.path.join(DATA_PATH, "modified_lstm_model.keras"))

## Display Training and Validation Curves

In [ ]:
import matplotlib.pyplot as plt

# Plot training & validation loss and accuracy
plt.figure(figsize=(12, 4))

# Loss curve
plt.subplot(1, 2, 1)
plt.plot(history_lstm_new.history['loss'], label='Training Loss')
plt.plot(history_lstm_new.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

# Accuracy curve
plt.subplot(1, 2, 2)
plt.plot(history_lstm_new.history['accuracy'], label='Training Accuracy')
plt.plot(history_lstm_new.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.tight_layout()
plt.show()

Evaluation of new LSTM

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
import seaborn as sns
import matplotlib.pyplot as plt

# Evaluate on test set
test_loss_lstm_new, test_acc_lstm_new = lstm_model_new.evaluate(X_test, y_test, verbose=0)
print(f"Test Accuracy (LSTM with modifications): {test_acc_lstm_new:.4f}")
print(f"Test Loss (LSTM with modifications): {test_loss_lstm_new:.4f}")

# Predictions
y_pred_probs = lstm_model_new.predict(X_test, verbose=0)
y_pred_lstm_new = np.argmax(y_pred_probs, axis=1)

# Metrics
accuracy_bn = accuracy_score(y_test, y_pred_lstm_new)
precision_bn = precision_score(y_test, y_pred_lstm_new, average='weighted')
recall_bn = recall_score(y_test, y_pred_lstm_new, average='weighted')
f1_bn = f1_score(y_test, y_pred_lstm_new, average='weighted')

print("\nDetailed Test Metrics (modified LSTM):")
print(f"Accuracy:  {accuracy_bn:.4f}")
print(f"Precision: {precision_bn:.4f}")
print(f"Recall:    {recall_bn:.4f}")
print(f"F1-Score:  {f1_bn:.4f}")

# Classification report
print("\nClassification Report (LSTM with modifications):")
print(classification_report(y_test, y_pred_lstm_new))

# Confusion matrix
cm_bn = confusion_matrix(y_test, y_pred_lstm_new)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_bn, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix - Modified LSTM Composer Classification')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()

## CNN Optimization:

optimization attempt here

In [ ]:
from tensorflow.keras.layers import BatchNormalization

cnn_model_bn = keras.Sequential([
    layers.Conv1D(64, kernel_size=7, activation='relu', input_shape=input_shape),
    layers.BatchNormalization(),
    layers.MaxPooling1D(pool_size=2),
    layers.Conv1D(128, kernel_size=7, activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling1D(pool_size=2),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(num_classes, activation='softmax')
])

cnn_model_bn.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

cnn_model_bn.summary()

# Train
history_cnn_bn = cnn_model_bn.fit(
    X_train, y_train,
    epochs=30,
    batch_size=16,
    validation_data=(X_dev, y_dev),
    callbacks=[early_stopping],
    verbose=0   # No verbose output for cleaner logs
)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
import seaborn as sns
import matplotlib.pyplot as plt

# Evaluate on test set
test_loss_cnn_bn, test_acc_cnn_bn = cnn_model_bn.evaluate(X_test, y_test, verbose=0)
print(f"Test Accuracy (CNN with BN): {test_acc_cnn_bn:.4f}")
print(f"Test Loss (CNN with BN): {test_loss_cnn_bn:.4f}")

# Predictions
y_pred_probs_cnn_bn = cnn_model_bn.predict(X_test, verbose=0)
y_pred_cnn_bn = np.argmax(y_pred_probs_cnn_bn, axis=1)

# Metrics
accuracy_cnn_bn = accuracy_score(y_test, y_pred_cnn_bn)
precision_cnn_bn = precision_score(y_test, y_pred_cnn_bn, average='weighted')
recall_cnn_bn = recall_score(y_test, y_pred_cnn_bn, average='weighted')
f1_cnn_bn = f1_score(y_test, y_pred_cnn_bn, average='weighted')

print("\nDetailed Test Metrics (CNN with BN):")
print(f"Accuracy:  {accuracy_cnn_bn:.4f}")
print(f"Precision: {precision_cnn_bn:.4f}")
print(f"Recall:    {recall_cnn_bn:.4f}")
print(f"F1-Score:  {f1_cnn_bn:.4f}")

# Classification report
print("\nClassification Report (CNN with BN):")
print(classification_report(y_test, y_pred_cnn_bn))

# Confusion matrix
cm_cnn_bn = confusion_matrix(y_test, y_pred_cnn_bn)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_cnn_bn, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix - CNN with BN Composer Classification')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()

## Ensemble Method (LSTM and CNN)

**Sugestions if we need**

Top-k accuracy (useful if you want to accept the correct class in top-2 or top-3 predictions)

In [ ]:
top_k_acc = tf.keras.metrics.top_k_categorical_accuracy(
    keras.utils.to_categorical(y_test, num_classes),
    lstm_model.predict(X_test), k=3
).numpy().mean()
print(f"Top-3 Accuracy: {top_k_acc:.3f}")


Feature Importance (Optional/Advanced)
You might use SHAP or similar tools for neural nets to see which note features most influence predictions (advanced).

Composer-level Metrics: Show metrics aggregated at the composer level (e.g., which composers are easiest/hardest to classify).

Error Analysis: Highlight and interpret common misclassifications—are certain composer pairs often confused? Why?

Experiment Tracker Table: Keep a markdown table of experiment settings/validation/test results to document LSTM vs. CNN runs.

Tempo/Key Analysis: Compare results on slow vs. fast, major vs. minor works (if you have such features).

Augmentation Studies: Try augmenting MIDI data (pitch/time shift) and study its impact (if time allows).

Notebook Polish: Add markdown explanations and section headers; use markdown cells to explain each experimental block.

Reproducibility: Save all random seeds and versions of key libraries used.

Conclusion Section: Summarize best results, strengths, and limitations; suggest future work (e.g., transformer models, including raw audio).